# 13 — Reshaping & Pivoting: Pivot Tables, Melt, Stack & Unstack
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive, battle-tested reference for Python Data Science, Business Intelligence, and ETL Interviews.*

---

## 📌 Executive Summary & Interview Expectations
Reshaping data between **wide format** (presentation-ready reports) and **long format** (machine learning & database-ready tidy data) is one of the most frequently tested concepts in senior data interviews.

### Core Competencies Tested in this Module:
1. **`pivot()` vs `pivot_table()`**: Why `pivot()` crashes on duplicate index keys with `ValueError` and why `pivot_table()` aggregates them.
2. **Multi-Metric Pivot Tables**: Custom `aggfunc` dictionaries, `margins=True` for grand totals, and `fill_value`.
3. **Stacking & Unstacking**: Reshaping along the MultiIndex axis (`stack()` wide $\to$ long, `unstack()` long $\to$ wide).
4. **Melting (`pd.melt`)**: The universal wide-to-long transformation for data normalization.
5. **Cross-Tabulation (`pd.crosstab`)**: Computing frequency distributions with `normalize='index'`.
6. **Interview Corner**: The Tidy Data Principle, reversing pivot tables, and multidimensional aggregation drills.

## 1. Environment Setup & Data Ingestion

In [1]:
import os
import numpy as np
import pandas as pd

# Load sales dataset
sales_path = "sales_by_employee.csv"
if not os.path.exists(sales_path):
    sales_path = "https://raw.githubusercontent.com/paskhaver/pandas-in-action/master/chapter_08_reshaping_and_pivoting/sales_by_employee.csv"

sales = pd.read_csv(sales_path, parse_dates=["Date"])
print("Sales dataset loaded. Shape:", sales.shape)
sales.head()

Sales dataset loaded. Shape: (26, 5)


/var/folders/54/5j97z6452x19yddj6yv2l7j00000gn/T/ipykernel_26819/1489023525.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sales = pd.read_csv(sales_path, parse_dates=["Date"])


,Date,Name,Customer,Revenue,Expenses
0,2020-01-01,Oscar,Logistics XYZ,5250,531
1,2020-01-01,Oscar,Money Corp.,4406,661
2,2020-01-02,Oscar,PaperMaven,8661,1401
3,2020-01-03,Oscar,PaperGenius,7075,906
4,2020-01-04,Oscar,Paper Pound,2524,1767


## 2. Pivot Tables: Aggregation Across Multiple Dimensions

### ⚠️ Top Interview Question: `pivot()` vs `pivot_table()`
| Feature | `df.pivot()` | `df.pivot_table()` |
| :--- | :--- | :--- |
| **Duplicate Index/Column Pairs** | **Crashes** (`ValueError: Index contains duplicate entries`) | **Aggregates** smoothly via `aggfunc` |
| **Aggregation Function** | None (pure reshape) | Default: `'mean'` (accepts `'sum'`, `'count'`, dicts) |
| **Missing Values** | Keeps `NaN` | Replaces with `fill_value=...` |
| **Subtotals & Margins** | Not supported | Supported via `margins=True` |

In [2]:
# Pivot: Total Revenue per Date by Salesperson
sales_pivot = sales.pivot_table(
    index="Date",
    columns="Name",
    values="Revenue",
    aggfunc="sum",
    fill_value=0
)
sales_pivot.head()

Name,Creed,Dwight,Jim,Michael,Oscar
Date,,,,,
2020-01-01,4430,2639,1864,7172,9656
2020-01-02,13214,0,8278,6362,8661
2020-01-03,0,11912,4226,5982,7075
2020-01-04,3144,0,6155,7917,2524
2020-01-05,938,7771,0,7837,2793


In [3]:
# Pivot with multiple aggregations: Mean and Sum of Revenue per Salesperson
multi_agg = sales.pivot_table(
    index="Date",
    columns="Name",
    values="Revenue",
    aggfunc=["sum", "mean"]
)
multi_agg.head(3)

sum                                     mean               \
Name          Creed   Dwight     Jim Michael   Oscar   Creed       Dwight   
Date                                                                        
2020-01-01   4430.0   2639.0  1864.0  7172.0  9656.0  4430.0  2639.000000   
2020-01-02  13214.0      NaN  8278.0  6362.0  8661.0  6607.0          NaN   
2020-01-03      NaN  11912.0  4226.0  5982.0  7075.0     NaN  3970.666667   

                                    
Name           Jim Michael   Oscar  
Date                                
2020-01-01  1864.0  7172.0  4828.0  
2020-01-02  8278.0  6362.0  8661.0  
2020-01-03  4226.0  5982.0  7075.0

## 3. Grand Totals & Subtotals with `margins=True`

In [4]:
# Pivot table with row and column totals (margins)
totals_pivot = sales.pivot_table(
    index="Date",
    columns="Name",
    values="Revenue",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total_Revenue"
)
totals_pivot.tail(4)

Name,Creed,Dwight,Jim,Michael,Oscar,Total_Revenue
Date,,,,,,
2020-01-03 00:00:00,0,11912,4226,5982,7075,29195
2020-01-04 00:00:00,3144,0,6155,7917,2524,19740
2020-01-05 00:00:00,938,7771,0,7837,2793,19339
Total_Revenue,21726,22322,20523,35270,30709,130550


## 4. Reshaping MultiIndex: `stack()` and `unstack()`

### 💡 Interview Mental Model:
- **`unstack()`**: Unstacks the inner row index to become columns (**Wider**).
- **`stack()`**: Stacks column headers into inner row indices (**Longer / Taller**).

In [5]:
# Start with a MultiIndex Series
grouped_sales = sales.groupby(["Date", "Name"])["Revenue"].sum()
print("Original MultiIndex Series (Long Format):")
display(grouped_sales.head(6))

Original MultiIndex Series (Long Format):


Date        Name   
2020-01-01  Creed       4430
            Dwight      2639
            Jim         1864
            Michael     7172
            Oscar       9656
2020-01-02  Creed      13214
Name: Revenue, dtype: int64

In [6]:
# Unstack: moves 'Name' from row index to column headers (Wide Format)
unstacked_df = grouped_sales.unstack(level="Name", fill_value=0)
print("Unstacked DataFrame (Wide Format):")
display(unstacked_df.head(3))

Unstacked DataFrame (Wide Format):


Name,Creed,Dwight,Jim,Michael,Oscar
Date,,,,,
2020-01-01,4430,2639,1864,7172,9656
2020-01-02,13214,0,8278,6362,8661
2020-01-03,0,11912,4226,5982,7075


In [7]:
# Stack: pivots columns back into row index (restores Long Format)
restacked_series = unstacked_df.stack()
print("Restacked back to Long Format:")
display(restacked_series.head(6))

Restacked back to Long Format:


Date        Name   
2020-01-01  Creed       4430
            Dwight      2639
            Jim         1864
            Michael     7172
            Oscar       9656
2020-01-02  Creed      13214
dtype: int64

## 5. Melting: Converting Wide Data to Tidy Long Format (`pd.melt`)

### ⚠️ Top Interview Question: Why do we Melt Data?
Machine learning algorithms (`scikit-learn`), SQL databases, and visualization libraries (`seaborn`, `plotly`) require **tidy data**:
1. Each variable forms a column.
2. Each observation forms a row.
`pd.melt()` unpivots wide columns into an identifier-variable-value schema.

In [8]:
# Load video game sales dataset
vg_path = "video_game_sales.csv"
if not os.path.exists(vg_path):
    vg_path = "https://raw.githubusercontent.com/paskhaver/pandas-in-action/master/chapter_08_reshaping_and_pivoting/video_game_sales.csv"

vg = pd.read_csv(vg_path)
print("Video game sales loaded. Columns:", list(vg.columns))
vg.head(2)

Video game sales loaded. Columns: ['Name', 'NA', 'EU', 'JP', 'Other']


,Name,NA,EU,JP,Other
0,Wii Sports,41.49,29.02,3.77,8.46
1,Super Mario Bros.,29.08,3.58,6.81,0.77


In [9]:
# Melt regional sales columns (NA, EU, JP, Other) into a single 'Region' feature
melted_vg = pd.melt(
    vg,
    id_vars=["Name"],
    value_vars=["NA", "EU", "JP", "Other"],
    var_name="Region",
    value_name="Sales_Millions"
)
print("Melted Long Format DataFrame:")
display(melted_vg.head(6))

Melted Long Format DataFrame:


,Name,Region,Sales_Millions
0,Wii Sports,NA,41.49
1,Super Mario Bros.,NA,29.08
2,Mario Kart Wii,NA,15.85
3,Wii Sports Resort,NA,15.75
4,Pokemon Red/Pokemon Blue,NA,11.27
5,Tetris,NA,23.20


## 6. Categorical Cross-Tabulation with `pd.crosstab`

In [10]:
# Load used cars dataset
cars = pd.read_csv("used_cars.csv")

# Frequency table between Fuel and Transmission
pd.crosstab(cars["Fuel"], cars["Transmission"], margins=True)

Transmission,Automatic,Manual,All
Fuel,,,
Diesel,11,4,15
Electric,4,6,10
Gas,5,10,15
Hybrid,8,2,10
All,28,22,50


In [11]:
# Normalized percentages across rows (proportions per fuel type)
pd.crosstab(cars["Fuel"], cars["Transmission"], normalize="index") * 100

Transmission,Automatic,Manual
Fuel,,
Diesel,73.333333,26.666667
Electric,40.000000,60.000000
Gas,33.333333,66.666667
Hybrid,80.000000,20.000000


## 7. Reshaping Cheat Sheet

| Operation | Function / Method | Transformation Direction | Key Parameters |
| :--- | :--- | :--- | :--- |
| **Pivot with Aggregation** | `df.pivot_table()` | Long $\to$ Wide | `index`, `columns`, `values`, `aggfunc`, `margins` |
| **Pivot without Aggregation** | `df.pivot()` | Long $\to$ Wide | Requires strictly unique index pairs |
| **Unstack** | `s.unstack(level)` | Rows $\to$ Columns | `level`, `fill_value` |
| **Stack** | `df.stack()` | Columns $\to$ Rows | Moves column header into inner row level |
| **Melt / Unpivot** | `pd.melt(df)` | Wide $\to$ Long | `id_vars`, `value_vars`, `var_name`, `value_name` |
| **Cross-Tab** | `pd.crosstab(s1, s2)` | 2D Contingency Table | `normalize={'all', 'index', 'columns'}` |

---
## 🎯 8. Technical Interview Corner: Tricky Questions & Drills

### Q1: The `ValueError` on Duplicate Entries in `pivot()`
**Question**: A candidate wrote `df.pivot(index='Date', columns='Store', values='Sales')` and Python crashed with:
`ValueError: Index contains duplicate entries, cannot reshape`.
Why did this happen and how do you fix it?

**Answer**:
- `df.pivot()` cannot aggregate. If a single store has 2 sales on the same date, Pandas cannot determine which value belongs in that cell.
- **Fix**: Use `df.pivot_table(index='Date', columns='Store', values='Sales', aggfunc='sum')` to aggregate duplicate transactions into a single value.

In [12]:
# Demonstration of duplicate entry handling
dup_demo = pd.DataFrame({
    "Date": ["2024-01-01", "2024-01-01", "2024-01-02"],
    "Store": ["A", "A", "A"],
    "Sales": [100, 150, 200]
})

# ✅ Using pivot_table aggregates duplicates cleanly:
display(dup_demo.pivot_table(index="Date", columns="Store", values="Sales", aggfunc="sum"))

Store,A
Date,
2024-01-01,250
2024-01-02,200


### Q2: Reversing a Pivot Table
**Question**: How do you invert a wide pivot table back into its original normalized tabular form?

**Answer**:
1. Reset the index: `pivoted_df.reset_index()`.
2. Use `pd.melt()` on the resulting DataFrame, setting the original index column as `id_vars`.

In [13]:
# Inverting sales_pivot back to original long form
inverted_sales = (
    sales_pivot.reset_index()
    .melt(id_vars=["Date"], var_name="Name", value_name="Revenue")
    .query("Revenue > 0")
    .sort_values(by=["Date", "Name"])
    .reset_index(drop=True)
)

print("Inverted Back to Long Form:")
display(inverted_sales.head(4))

Inverted Back to Long Form:


,Date,Name,Revenue
0,2020-01-01,Creed,4430
1,2020-01-01,Dwight,2639
2,2020-01-01,Jim,1864
3,2020-01-01,Michael,7172


### Q3: Advanced Interview Challenge: Top Region per Video Game
**Challenge**: Given the melted video game sales dataset, find the **dominant sales region** (highest revenue) for each video game title in a single chained expression!

In [14]:
# Solution to Coding Challenge using sort_values and drop_duplicates
top_region_per_game = (
    melted_vg.sort_values(by="Sales_Millions", ascending=False)
    .drop_duplicates(subset=["Name"])
    .rename(columns={"Region": "Top_Region", "Sales_Millions": "Peak_Sales_M"})
    .head(5)
)

print("Dominant sales regions for top titles:")
display(top_region_per_game)

Dominant sales regions for top titles:


,Name,Top_Region,Peak_Sales_M
0,Wii Sports,NA,41.49
1,Super Mario Bros.,NA,29.08
9,Duck Hunt,NA,26.93
5,Tetris,NA,23.20
2,Mario Kart Wii,NA,15.85
